In [1]:
# ==========================
# 1. Configuração
# ==========================
from pathlib import Path
import platform
import sqlite3
import numpy as np
import pandas as pd

if platform.system() == 'Windows':
    DB_PATH = Path(r'C:\Users\LISARR\Documents\python\01.Financeiro\inform_27.db')
    DANONE_PATH = Path(r'C:\Users\LISARR\Documents\python\01.Financeiro\Danone_Custos_V2.xlsx')
    COORDS_PATH = Path(r'C:\Users\LISARR\Documents\python\00.Metadados\CP7COORDS_FORMATADO.CSV')
elif platform.system() == 'Darwin':
    DB_PATH = Path('/Users/rr/Library/Mobile Documents/com~apple~CloudDocs/05.Salvesen/inform_27.db')
    DANONE_PATH = DB_PATH.parent / 'Danone_Custos_V2.xlsx'
    COORDS_PATH = DB_PATH.parent / 'CP7COORDS_FORMATADO.CSV'
else:
    DB_PATH = Path('inform_27.db')
    DANONE_PATH = Path('Danone_Custos_V2.xlsx')
    COORDS_PATH = Path('CP7COORDS_FORMATADO.CSV')

OUTPUT_PATH = DB_PATH.parent / 'inform_27_2026_final.parquet'
TABELA = 'inform_27_2026'


In [2]:
# ==========================
# 2. Leitura da base
# ==========================
with sqlite3.connect(DB_PATH) as con:
    df = pd.read_sql_query(f'SELECT * FROM "{TABELA}"', con)

linhas_iniciais = len(df)

df


,id,PROPIETARIO,TRAYECTO,TRANSPORTISTA,TRACTORA,REMOLQUE,INGRESODT,COSTEDT,RENTADT,PALETSDT,...,LOCDES,LUGARDESCARGA,TEMP_MERC_PED,TIPOPALETA,CAMION_TIPO,CAMION_CAPACIDAD,TIPO_COMBUSTIBLE,KMREALES,ALBARAN,ficheiro_origem
0,1,4PL,CARREFOUR BOLLENE / PLAT. CARREFOUR MIRALCAMPO,CENTROTIR LOGISTIC SERVICES S.A.,3131-MST,0000XXX,0,0,0,0.11,...,297643,PLAT. CARREFOUR MIRALCAMPO,RFG,EUR,Trailer 33 plts,33,DIESEL,1014.971,NO,SAL_DAT027 (1).xls
1,2,4PL,CARREFOUR BOLLENE / PLAT. CARREFOUR MIRALCAMPO,CENTROTIR LOGISTIC SERVICES S.A.,3131-MST,0000XXX,0,0,0,0.4,...,297643,PLAT. CARREFOUR MIRALCAMPO,RFG,EUR,Trailer 33 plts,33,DIESEL,1014.971,NO,SAL_DAT027 (1).xls
2,3,4PL,CARREFOUR BOLLENE / PLAT. CARREFOUR MIRALCAMPO,CENTROTIR LOGISTIC SERVICES S.A.,3131-MST,0000XXX,0,0,0,1,...,297643,PLAT. CARREFOUR MIRALCAMPO,RFG,EUR,Trailer 33 plts,33,DIESEL,1014.971,NO,SAL_DAT027 (1).xls
3,4,4PL,CARREFOUR BOLLENE / PLAT. CARREFOUR MIRALCAMPO,CENTROTIR LOGISTIC SERVICES S.A.,3131-MST,0000XXX,0,0,0,1,...,297643,PLAT. CARREFOUR MIRALCAMPO,RFG,EUR,Trailer 33 plts,33,DIESEL,1014.971,NO,SAL_DAT027 (1).xls
4,5,4PL,CARREFOUR BOLLENE / PLAT. CARREFOUR MIRALCAMPO,CENTROTIR LOGISTIC SERVICES S.A.,3131-MST,0000XXX,0,0,0,1.02,...,297643,PLAT. CARREFOUR MIRALCAMPO,RFG,EUR,Trailer 33 plts,33,DIESEL,1014.971,NO,SAL_DAT027 (1).xls
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
756016,3825406,BCN,DL TRANSMEDITERRANEA BARCELONA / DINOSOL SUPER...,TIERS SPOT,,GENERICO,254.32,0,254.32,5.15,...,574677,DL TRANSMEDITERRANEA BARCELONA,RFGRFG,EUR,GENÉRICO,0,DIESEL,0,NO,SAL_DAT027_2025_07_12.csv
756017,3825407,BCN,DL TRANSMEDITERRANEA BARCELONA / DINOSOL SUPER...,TIERS SPOT,,GENERICO,254.32,0,254.32,5.15,...,167823,"DINOSOL SUPERMERCADOS, S.L.",RFGRFG,EUR,GENÉRICO,0,DIESEL,1273.709,NO,SAL_DAT027_2025_07_12.csv
756018,3825412,BCN,DL TRANSMEDITERRANEA BARCELONA / DINOSOL SUPER...,TIERS SPOT,,GENERICO,0,0,0,0.35,...,574677,DL TRANSMEDITERRANEA BARCELONA,RFGRFG,EUR,GENÉRICO,0,DIESEL,0,NO,SAL_DAT027_2025_07_12.csv
756019,3825413,BCN,DL TRANSMEDITERRANEA BARCELONA / DINOSOL SUPER...,TIERS SPOT,,GENERICO,0,0,0,0.35,...,635578,"AGRUCAN, S.L.",RFGRFG,EUR,GENÉRICO,0,DIESEL,1282.699,NO,SAL_DAT027_2025_07_12.csv


In [3]:
# ==========================
# 3. Tipos e limpeza
# ==========================
colunas_numericas = [
    'INGRESODT', 'COSTEDT', 'RENTADT', 'PALETSDT',
    'PESO_BRUTO', 'PALETS', 'KM', 'KMREALES',
]

for coluna in colunas_numericas:
    if coluna in df.columns:
        df[coluna] = pd.to_numeric(df[coluna], errors='coerce')

for coluna in df.select_dtypes(include=['object', 'string']).columns:
    df[coluna] = df[coluna].astype('string').str.strip()

df['CODEUT'] = pd.to_numeric(df['CODEUT'], errors='coerce').astype('Int64').astype('string')
df['CODEDT'] = pd.to_numeric(df['CODEDT'], errors='coerce').astype('Int64').astype('string')
df['REFERENCIA'] = df['REFERENCIA'].astype('string').str.strip()

df['data_carga'] = pd.to_datetime(df['FCARGA'], format='%Y%m%d', errors='coerce')
df['data_entrega'] = pd.to_datetime(df['FENTREGA'], format='%Y%m%d', errors='coerce')
df['data'] = df['data_entrega']

df.dtypes


id                          int64
PROPIETARIO        string[python]
TRAYECTO           string[python]
TRANSPORTISTA      string[python]
TRACTORA           string[python]
                        ...      
ALBARAN            string[python]
ficheiro_origem    string[python]
data_carga         datetime64[ns]
data_entrega       datetime64[ns]
data               datetime64[ns]
Length: 61, dtype: object

In [4]:
# ==========================
# 4. Calendário
# ==========================
meses = {
    1: 'Janeiro', 2: 'Fevereiro', 3: 'Março', 4: 'Abril',
    5: 'Maio', 6: 'Junho', 7: 'Julho', 8: 'Agosto',
    9: 'Setembro', 10: 'Outubro', 11: 'Novembro', 12: 'Dezembro',
}

dias = {
    1: 'Segunda-feira', 2: 'Terça-feira', 3: 'Quarta-feira',
    4: 'Quinta-feira', 5: 'Sexta-feira', 6: 'Sábado', 7: 'Domingo',
}

iso = df['data'].dt.isocalendar()

df['ano'] = df['data'].dt.year
df['mes'] = df['data'].dt.month
df['mes_nome'] = df['mes'].map(meses)
df['ano_semana_iso'] = iso['year']
df['week_number'] = iso['week']
df['week_day_number'] = df['data'].dt.weekday + 1
df['week_day'] = df['week_day_number'].map(dias)

df.groupby(['ano', 'mes', 'mes_nome']).size().rename('linhas').reset_index()


,ano,mes,mes_nome,linhas
0,2026,1,Janeiro,99110
1,2026,2,Fevereiro,95004
2,2026,3,Março,105638
3,2026,4,Abril,100617
4,2026,5,Maio,96008
5,2026,6,Junho,97647
6,2026,7,Julho,100334
7,2026,8,Agosto,61663


In [5]:
# ==========================
# 5. Ingresso Danone
# ==========================
def limpar_referencia(serie):
    return (
        serie
        .astype('string')
        .str.extract(
            r'(\d{10})',
            expand=False,
        )
        .replace('', pd.NA)
    )


# ==========================
# 5.1. Limpeza anterior
# ==========================
colunas_anteriores = [
    'REFERENCIA_ORIGINAL',
    'REFERENCIA_CHAVE',
    'REFERENCIA_KILOSPO',
    'ingresso_danone',
    'total_ingresso',
]

df = df.drop(
    columns=[
        coluna
        for coluna in colunas_anteriores
        if coluna in df.columns
    ]
)


# ==========================
# 5.2. Leitura do KILOSPO
# ==========================
df_danone = pd.read_excel(
    DANONE_PATH,
    sheet_name='KILOSPO',
    usecols=[
        'PREFPE',
        'PFEENT',
        'Combustible',
    ],
)

df_danone.columns = [
    'REFERENCIA_KILOSPO',
    'data_entrega',
    'ingresso_danone',
]


# ==========================
# 5.3. Tratamento do KILOSPO
# ==========================
df_danone['REFERENCIA_KILOSPO'] = (
    df_danone['REFERENCIA_KILOSPO']
    .astype('string')
    .str.strip()
)

df_danone['REFERENCIA_CHAVE'] = limpar_referencia(
    df_danone['REFERENCIA_KILOSPO']
)

df_danone['data_entrega'] = pd.to_datetime(
    pd.to_numeric(
        df_danone['data_entrega'],
        errors='coerce',
    )
    .astype('Int64')
    .astype('string'),
    format='%Y%m%d',
    errors='coerce',
)

df_danone['ingresso_danone'] = pd.to_numeric(
    df_danone['ingresso_danone'],
    errors='coerce',
)

df_danone = (
    df_danone
    .dropna(
        subset=[
            'REFERENCIA_CHAVE',
            'data_entrega',
            'ingresso_danone',
        ]
    )
    .groupby(
        [
            'REFERENCIA_CHAVE',
            'data_entrega',
        ],
        as_index=False,
    )
    .agg(
        REFERENCIA_KILOSPO=(
            'REFERENCIA_KILOSPO',
            'first',
        ),
        ingresso_danone=(
            'ingresso_danone',
            'sum',
        ),
    )
)


# ==========================
# 5.4. Tratamento da base
# ==========================
df['REFERENCIA_ORIGINAL'] = (
    df['REFERENCIA']
    .astype('string')
)

df['REFERENCIA_CHAVE'] = limpar_referencia(
    df['REFERENCIA_ORIGINAL']
)

df['data_entrega'] = pd.to_datetime(
    df['data_entrega'],
    errors='coerce',
).dt.normalize()

df['_ordem_original'] = range(len(df))


# ==========================
# 5.5. Correspondência exata
# ==========================
df = df.merge(
    df_danone,
    on=[
        'REFERENCIA_CHAVE',
        'data_entrega',
    ],
    how='left',
    validate='many_to_one',
)

repetida_mesmo_dia = (
    df['ingresso_danone'].notna()
    & df.duplicated(
        subset=[
            'REFERENCIA_CHAVE',
            'data_entrega',
        ],
        keep='first',
    )
)

df.loc[
    repetida_mesmo_dia,
    'ingresso_danone',
] = 0


# ==========================
# 5.6. Registos sem data exata
# ==========================
chaves_df = (
    df[
        [
            'REFERENCIA_CHAVE',
            'data_entrega',
        ]
    ]
    .dropna()
    .drop_duplicates()
)

kilospo_sem_data_exata = df_danone.merge(
    chaves_df,
    on=[
        'REFERENCIA_CHAVE',
        'data_entrega',
    ],
    how='left',
    indicator=True,
)

kilospo_sem_data_exata = (
    kilospo_sem_data_exata
    .loc[
        kilospo_sem_data_exata[
            '_merge'
        ].eq('left_only')
    ]
    .drop(columns='_merge')
    .copy()
)


# ==========================
# 5.7. Fallback por referência
# ==========================
referencias_df = set(
    df['REFERENCIA_CHAVE']
    .dropna()
    .unique()
)

fallback_referencia = (
    kilospo_sem_data_exata
    .loc[
        kilospo_sem_data_exata[
            'REFERENCIA_CHAVE'
        ].isin(referencias_df)
    ]
    .groupby('REFERENCIA_CHAVE')[
        'ingresso_danone'
    ]
    .sum()
)

ultima_linha_referencia = ~df.duplicated(
    subset=['REFERENCIA_CHAVE'],
    keep='last',
)

valor_fallback = df[
    'REFERENCIA_CHAVE'
].map(
    fallback_referencia
)

aplicar_fallback = (
    ultima_linha_referencia
    & valor_fallback.notna()
)

df.loc[
    aplicar_fallback,
    'ingresso_danone',
] = (
    df.loc[
        aplicar_fallback,
        'ingresso_danone',
    ].fillna(0)
    + valor_fallback.loc[
        aplicar_fallback
    ]
)


# ==========================
# 5.8. Resultado final
# ==========================
df = (
    df
    .sort_values('_ordem_original')
    .drop(columns='_ordem_original')
    .reset_index(drop=True)
)

df['total_ingresso'] = (
    df['INGRESODT'].fillna(0)
    + df['ingresso_danone'].fillna(0)
)


# ==========================
# 5.9. Registos inexistentes
# ==========================
referencias_inexistentes = (
    kilospo_sem_data_exata
    .loc[
        ~kilospo_sem_data_exata[
            'REFERENCIA_CHAVE'
        ].isin(referencias_df)
    ]
    .copy()
)


# ==========================
# 5.10. Validação
# ==========================
ingresso_kilospo = df_danone[
    'ingresso_danone'
].sum()

ingresso_mapeado = df[
    'ingresso_danone'
].sum()

ingresso_nao_mapeado = referencias_inexistentes[
    'ingresso_danone'
].sum()

resumo_danone = pd.DataFrame({
    'indicador': [
        'Ingresso KILOSPO',
        'Ingresso mapeado',
        'Ingresso não mapeado',
        'Taxa financeira mapeada (%)',
        'Repetições no mesmo dia',
        'Fallback aplicado na última linha',
        'Referências ainda inexistentes',
    ],
    'valor': [
        ingresso_kilospo,
        ingresso_mapeado,
        ingresso_nao_mapeado,
        (
            ingresso_mapeado
            / ingresso_kilospo
            * 100
        ),
        repetida_mesmo_dia.sum(),
        aplicar_fallback.sum(),
        referencias_inexistentes[
            'REFERENCIA_CHAVE'
        ].nunique(),
    ],
})

resumo_danone

,indicador,valor
0,Ingresso KILOSPO,1.029127e+06
1,Ingresso mapeado,1.024176e+06
2,Ingresso não mapeado,4.950974e+03
3,Taxa financeira mapeada (%),9.951892e+01
4,Repetições no mesmo dia,3.324000e+03
5,Fallback aplicado na última linha,2.240000e+02
6,Referências ainda inexistentes,1.933000e+03


In [6]:
df.head()

,id,PROPIETARIO,TRAYECTO,TRANSPORTISTA,TRACTORA,REMOLQUE,INGRESODT,COSTEDT,RENTADT,PALETSDT,...,mes_nome,ano_semana_iso,week_number,week_day_number,week_day,REFERENCIA_ORIGINAL,REFERENCIA_CHAVE,REFERENCIA_KILOSPO,ingresso_danone,total_ingresso
0,1,4PL,CARREFOUR BOLLENE / PLAT. CARREFOUR MIRALCAMPO,CENTROTIR LOGISTIC SERVICES S.A.,3131-MST,0000XXX,0.0,0.0,0.0,0.11,...,Fevereiro,2026,6,3,Quarta-feira,H269269097,<NA>,<NA>,NaN,0.0
1,2,4PL,CARREFOUR BOLLENE / PLAT. CARREFOUR MIRALCAMPO,CENTROTIR LOGISTIC SERVICES S.A.,3131-MST,0000XXX,0.0,0.0,0.0,0.40,...,Fevereiro,2026,6,3,Quarta-feira,H269269123,<NA>,<NA>,NaN,0.0
2,3,4PL,CARREFOUR BOLLENE / PLAT. CARREFOUR MIRALCAMPO,CENTROTIR LOGISTIC SERVICES S.A.,3131-MST,0000XXX,0.0,0.0,0.0,1.00,...,Fevereiro,2026,6,3,Quarta-feira,H269269137,<NA>,<NA>,NaN,0.0
3,4,4PL,CARREFOUR BOLLENE / PLAT. CARREFOUR MIRALCAMPO,CENTROTIR LOGISTIC SERVICES S.A.,3131-MST,0000XXX,0.0,0.0,0.0,1.00,...,Fevereiro,2026,6,3,Quarta-feira,H269269154,<NA>,<NA>,NaN,0.0
4,5,4PL,CARREFOUR BOLLENE / PLAT. CARREFOUR MIRALCAMPO,CENTROTIR LOGISTIC SERVICES S.A.,3131-MST,0000XXX,0.0,0.0,0.0,1.02,...,Fevereiro,2026,6,3,Quarta-feira,H269269173,<NA>,<NA>,NaN,0.0


# metricas


In [7]:
# ==========================
# 6. Capacidade e métricas
# ==========================
mapeamento_capacidade = {
    4: 6, 5: 6, 6: 6,
    8: 12, 12: 12,
    14: 20, 15: 20, 16: 20, 18: 20, 20: 20,
    22: 24, 24: 24,
    33: 33, 66: 66,
}

df['capacidade_original'] = pd.to_numeric(df['CAMION_CAPACIDAD'], errors='coerce')
df['capacidade_norm'] = df['capacidade_original'].map(mapeamento_capacidade)

capacidade_codeut = df.groupby('CODEUT')['capacidade_norm'].transform('max')
df['capacidade_norm'] = df['capacidade_norm'].fillna(capacidade_codeut)

df['dados_validos'] = (
    df['PALETS'].notna()
    & df['PALETS'].ge(0)
    & df['capacidade_norm'].gt(0)
)

paletes_validas = df['PALETS'].replace(0, np.nan)
df['custo_por_palete'] = df['COSTEDT'] / paletes_validas
df['ingresso_por_palete'] = df['total_ingresso'] / paletes_validas
df['margem'] = df['total_ingresso'] - df['COSTEDT']
df['margem_por_palete'] = df['margem'] / paletes_validas

df['total_paletes_rota'] = df.groupby('CODEUT')['PALETS'].transform('sum')
df['taxa_ocupacao_rota'] = (
    df['total_paletes_rota'] / df['capacidade_norm'] * 100
)

df.loc[~df['dados_validos'], [
    'custo_por_palete', 'ingresso_por_palete',
    'margem_por_palete', 'taxa_ocupacao_rota',
]] = np.nan

df[['CODEUT', 'PALETS', 'capacidade_norm', 'total_ingresso', 'margem', 'taxa_ocupacao_rota']]


,CODEUT,PALETS,capacidade_norm,total_ingresso,margem,taxa_ocupacao_rota
0,3362794,1.0,33.0,0.00,0.00,136.363636
1,3362794,1.0,33.0,0.00,0.00,136.363636
2,3362794,1.0,33.0,0.00,0.00,136.363636
3,3362794,1.0,33.0,0.00,0.00,136.363636
4,3362794,2.0,33.0,0.00,0.00,136.363636
...,...,...,...,...,...,...
756016,3350263,6.0,NaN,254.32,254.32,NaN
756017,3350263,6.0,NaN,254.32,254.32,NaN
756018,3350263,1.0,NaN,0.00,0.00,NaN
756019,3350263,1.0,NaN,0.00,0.00,NaN


In [8]:
# ==========================
# 7. Coordenadas
# ==========================
coords = pd.read_csv(COORDS_PATH, sep=';', encoding='utf-8')

coords['CP'] = coords['CP'].astype('string').str.strip()
coords['POINT_X'] = pd.to_numeric(
    coords['POINT_X'].astype('string').str.replace(',', '.', regex=False),
    errors='coerce',
)
coords['POINT_Y'] = pd.to_numeric(
    coords['POINT_Y'].astype('string').str.replace(',', '.', regex=False),
    errors='coerce',
)

coords = (
    coords.dropna(subset=['CP', 'POINT_X', 'POINT_Y'])
    .groupby('CP', as_index=False)[['POINT_X', 'POINT_Y']]
    .mean()
)

coords_origem = coords.rename(columns={
    'CP': 'CPOSTAL',
    'POINT_X': 'longitude_origem',
    'POINT_Y': 'latitude_origem',
})
coords_destino = coords.rename(columns={
    'CP': 'CPOSTAD',
    'POINT_X': 'longitude_destino',
    'POINT_Y': 'latitude_destino',
})

df['CPOSTAL'] = df['CPOSTAL'].astype('string').str.strip()
df['CPOSTAD'] = df['CPOSTAD'].astype('string').str.strip()

df = df.merge(coords_origem, on='CPOSTAL', how='left', validate='many_to_one')
df = df.merge(coords_destino, on='CPOSTAD', how='left', validate='many_to_one')

pd.DataFrame({
    'local': ['Origem', 'Destino'],
    'matches': [
        df['longitude_origem'].notna().sum(),
        df['longitude_destino'].notna().sum(),
    ],
})


FileNotFoundError: [Errno 2] No such file or directory: '/Users/rr/Library/Mobile Documents/com~apple~CloudDocs/05.Salvesen/CP7COORDS_FORMATADO.CSV'

In [ ]:
# ==========================
# 8. Tabela final
# ==========================
colunas_finais = [
    'id', 'data', 'data_carga', 'data_entrega',
    'ano', 'mes', 'mes_nome', 'ano_semana_iso',
    'week_number', 'week_day_number', 'week_day',
    'CODEUT', 'CODEDT', 'REFERENCIA', 'CODACT', 'ACTIVIDAD',
    'TRANSPORTISTA', 'TRACTORA', 'CAMION_TIPO', 'CAMION_CAPACIDAD',
    'capacidade_original', 'capacidade_norm', 'PALETS', 'PESO_BRUTO',
    'INGRESODT', 'ingresso_danone', 'total_ingresso', 'COSTEDT',
    'custo_por_palete', 'ingresso_por_palete', 'margem',
    'margem_por_palete', 'total_paletes_rota', 'taxa_ocupacao_rota',
    'PROV_ORIGEN', 'LOCORIGEN', 'CPOSTAL',
    'longitude_origem', 'latitude_origem',
    'PROV_DESTINO', 'LOCDESTINO', 'CPOSTAD',
    'longitude_destino', 'latitude_destino',
    'PROV_ENTREGAR', 'PAISENTREGAR',
    'LOCCAR', 'LUGARCARGA', 'LOCDES', 'LUGARDESCARGA',
    'TIPOCLIENTE', 'TIPOFLUJO', 'dados_validos', 'ficheiro_origem',
]

colunas_finais = [coluna for coluna in colunas_finais if coluna in df.columns]
df_sel = df[colunas_finais].copy()

if len(df_sel) != linhas_iniciais:
    raise ValueError('O número de linhas foi alterado durante os cruzamentos.')

df_sel


In [ ]:
# ==========================
# 9. Validação e exportação
# ==========================
validacao = pd.DataFrame({
    'teste': [
        'Linhas preservadas',
        'Ingresso reconciliado',
        'Margem reconciliada',
    ],
    'resultado': [
        len(df_sel) == linhas_iniciais,
        np.isclose(
            df_sel['total_ingresso'].sum(),
            df_sel['INGRESODT'].fillna(0).sum()
            + df_sel['ingresso_danone'].fillna(0).sum(),
        ),
        np.isclose(
            df_sel['margem'].sum(),
            df_sel['total_ingresso'].sum() - df_sel['COSTEDT'].sum(),
        ),
    ],
})

if not validacao['resultado'].all():
    raise ValueError('A validação final falhou.')

df_sel.to_parquet(OUTPUT_PATH, index=False)

pd.DataFrame({
    'ficheiro': [str(OUTPUT_PATH)],
    'linhas': [len(df_sel)],
    'colunas': [len(df_sel.columns)],
    'tamanho_mb': [OUTPUT_PATH.stat().st_size / 1024**2],
})
